# Experiment 10: Layer 0 Submodule Tucker Adaptation — Nested Clustering & 1% Recon Sweep

**Target Model**: `google/gemma-3-1b-it` — `model.model.layers[0]`

### Exact Target Submodules (from `01_activation_profiling.ipynb`):

| Submodule | Output Shape | Tucker Target | Treatment |
|:---|:---:|:---:|:---|
| `self_attn` | `(5000, 1152)` | Output activation | Tucker on 3D reshaped activation tensor |
| `self_attn.q_proj` | `(5000, 1024)` | Weight matrix `[1024, 1152]` | Tucker on weight via nested clustering |
| `self_attn.k_proj` | `(5000, 256)` | Weight matrix `[256, 1152]` | Tucker on weight via nested clustering |
| `self_attn.v_proj` | `(5000, 256)` | Weight matrix `[256, 1152]` | Tucker on weight via nested clustering |
| `self_attn.o_proj` | `(5000, 1152)` | Weight matrix `[1152, 1024]` | Tucker on weight via nested clustering |
| `self_attn.q_norm` | `(5000, 256)` | Output activation | Tucker on 3D reshaped activation tensor |
| `self_attn.k_norm` | `(5000, 256)` | Output activation | Tucker on 3D reshaped activation tensor |
| `mlp` | `(5000, 1152)` | Output activation | Tucker on 3D reshaped activation tensor |

> **Note**: `mlp.gate_proj`, `mlp.up_proj`, `mlp.down_proj` are **NOT** individually targeted — `mlp` is treated as a single module via its output activation.

---

### Methodology:
1. **Normal Distribution Outlier Thresholding** ($|z| > 3.0$ or top 1% activation variance) — quarantines superactivations/superweights.
2. **Nested DBSCAN Clustering** ($N=3$ iterations) — partitions coordinates into balanced chunks to form $\mathcal{T} \in \mathbb{R}^{K \times M \times D}$.
3. **1% Incremental Reconstruction Rate Loss Sweep** — steps $\tau \in [0.99, 0.70]$ to find Pareto optimal Tucker ranks $[R_1, R_2, R_3]$.
4. **Qualitative Generation** — Chocolate cake recipe query evaluated at best rank composition per submodule.


In [ ]:
# =====================================================================
# STEP 1: Environment Setup & Imports
# =====================================================================
import os
import sys
import time
import json
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from datasets import load_dataset
from tqdm.auto import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from transformers import AutoModelForCausalLM, AutoTokenizer

# Point Triton cache to writable location
os.environ["TRITON_CACHE_DIR"] = os.path.expanduser("~/.triton_cache")
os.makedirs(os.environ["TRITON_CACHE_DIR"], exist_ok=True)
os.environ["HF_DATASETS_OFFLINE"] = "1"

tl.set_backend("pytorch")
torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB total")



In [ ]:
# =====================================================================
# STEP 2: Load Model & Tokenizer
# =====================================================================
MODEL_ID = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32, device_map=device)
model.eval()

l0 = model.model.layers[0]

# Verify FP32 loaded correctly (assertion fails loudly if dtype silently changed)
actual_dtype = next(model.parameters()).dtype
total_params = sum(x.numel() for x in model.parameters())
assert actual_dtype == torch.float32, f"Expected float32 but got {actual_dtype}"
print(f"Loaded {MODEL_ID}")
print(f"  dtype  : {actual_dtype}")
print(f"  params : {total_params:,}  ({total_params/1e9:.3f}B)")
print(f"  VRAM   : {total_params * 4 / 1024**3:.2f} GiB (weights-only estimate)")


In [ ]:
# =====================================================================
# STEP 3: Evaluation Helpers — MNLI & Chocolate Cake Recipe Query
# =====================================================================
EVAL_SAMPLES = 150

print(f"Loading GLUE MNLI validation_matched ({EVAL_SAMPLES} samples)...")
ds = load_dataset("nyu-mll/glue", "mnli", split="validation_matched").select(range(EVAL_SAMPLES))
labels_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + n, add_special_tokens=False)[0] for n in labels_names]

CAKE_PROMPT = (
    "<start_of_turn>user\n"
    "What is the best recipe to make a chocolate cake?<end_of_turn>\n"
    "<start_of_turn>model\n"
)

def evaluate_mnli(model) -> float:
    model.eval()
    preds, gt = [], []
    with torch.no_grad():
        for sample in ds:
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inp = tokenizer(prompt, return_tensors="pt").to(model.device)
            out = model(**inp, logits_to_keep=1)
            preds.append(torch.argmax(out.logits[0, -1, :][label_token_ids]).item())
            gt.append(sample["label"])
    return float(accuracy_score(gt, preds))

def generate_cake_recipe(model, max_new_tokens=256) -> str:
    model.eval()
    inp = tokenizer(CAKE_PROMPT, return_tensors="pt").to(model.device)
    with torch.no_grad():
        tokens = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9)
    return tokenizer.decode(tokens[0][inp.input_ids.shape[1]:], skip_special_tokens=True)



In [ ]:
# =====================================================================
# STEP 4: Forward Hook Profiler for the 8 Target Submodules
# =====================================================================
# We capture the OUTPUT activation of each of the 8 submodules.
# For Linear projections the activation is used both for clustering guidance
# and for the Tucker analysis.
# For container modules (self_attn, mlp) and norm layers (q_norm, k_norm),
# the activation tensor IS the Tucker decomposition target.

activation_store: Dict[str, List[torch.Tensor]] = {
    "self_attn":      [],   # container: captures self_attn output (5000, 1152)
    "self_attn.q_proj": [], # linear: captures q_proj output (5000, 1024)
    "self_attn.k_proj": [], # linear: captures k_proj output (5000, 256)
    "self_attn.v_proj": [], # linear: captures v_proj output (5000, 256)
    "self_attn.o_proj": [], # linear: captures o_proj output (5000, 1152)
    "self_attn.q_norm": [], # 1D RMSNorm: captures q_norm output (5000, 256)
    "self_attn.k_norm": [], # 1D RMSNorm: captures k_norm output (5000, 256)
    "mlp":            [],   # container: captures mlp output (5000, 1152)
}

def make_hook(name):
    def hook(module, inp, out):
        tensor = out[0] if isinstance(out, tuple) else out
        flat = tensor.detach().cpu().float().reshape(-1, tensor.shape[-1])
        activation_store[name].append(flat)
    return hook

hooks = [
    l0.self_attn.register_forward_hook(make_hook("self_attn")),
    l0.self_attn.q_proj.register_forward_hook(make_hook("self_attn.q_proj")),
    l0.self_attn.k_proj.register_forward_hook(make_hook("self_attn.k_proj")),
    l0.self_attn.v_proj.register_forward_hook(make_hook("self_attn.v_proj")),
    l0.self_attn.o_proj.register_forward_hook(make_hook("self_attn.o_proj")),
    l0.self_attn.q_norm.register_forward_hook(make_hook("self_attn.q_norm")),
    l0.self_attn.k_norm.register_forward_hook(make_hook("self_attn.k_norm")),
    l0.mlp.register_forward_hook(make_hook("mlp")),
]

# Baseline MNLI + activation collection
print("Running baseline MNLI evaluation and capturing activations...")
baseline_acc = evaluate_mnli(model)
for h in hooks:
    h.remove()

# Stack & trim to max 5000 tokens
acts: Dict[str, np.ndarray] = {}
for name, tensors in activation_store.items():
    if tensors:
        cat = torch.cat(tensors, dim=0).numpy()
        acts[name] = cat[:5000]

print(f"\nPristine Baseline MNLI Accuracy: {baseline_acc * 100:.2f}%\n")
print(f"{'Submodule':<24} | {'Captured Shape':<20} | {'Mean':>8} | {'Std':>8}")
print("-" * 70)
for name, a in acts.items():
    print(f"{name:<24} | {str(a.shape):<20} | {a.mean():>8.4f} | {a.std():>8.4f}")

# Baseline cake recipe
print("\nGenerating baseline chocolate cake recipe...")
baseline_cake = generate_cake_recipe(model)
print("\n--- Pristine Baseline Cake Recipe ---")
print(baseline_cake[:500], "...\n")



In [ ]:
# =====================================================================
# STEP 5: Normal Thresholding + Nested DBSCAN Clustering
# =====================================================================
def nested_clustering_with_thresholding(
    act_matrix: np.ndarray,    # shape (N_tokens, dim)
    chunk_size: int,
    num_chunks: int,
    n_iter: int = 3,
    z_cutoff: float = 3.0,
) -> Dict[str, Any]:
    """
    1. Computes mean, variance, z-score across coordinate (feature) axis.
    2. Quarantines superactivation coordinates (|z|>3 or top-1% variance).
    3. Iterative DBSCAN on remaining coordinates (n_iter passes on residuals).
    4. Returns balanced 3D tensor T (num_chunks x chunk_size x hidden_dim).
    """
    dim = act_matrix.shape[1]
    v = np.mean(act_matrix, axis=0)          # (dim,)
    variances = np.var(act_matrix, axis=0)   # (dim,)

    # Normal Distribution Thresholding
    z = np.abs((v - np.mean(v)) / (np.std(v) + 1e-8))
    var99 = float(np.quantile(variances, 0.99)) if dim > 10 else 1e9
    super_mask = (z > z_cutoff) | (variances >= var99)
    super_coords = np.where(super_mask)[0]

    candidate_idx = np.where(~super_mask)[0]

    # Dynamically shrink chunk_size if we don't have enough candidates
    eff_chunk = chunk_size
    if len(candidate_idx) < num_chunks * eff_chunk:
        eff_chunk = max(5, len(candidate_idx) // num_chunks)

    chunk_list = []

    # Nested DBSCAN clustering
    for _ in range(n_iter):
        if len(candidate_idx) < eff_chunk or len(chunk_list) >= num_chunks:
            break
        v_sub = v[candidate_idx]
        eps = max(0.02, float(np.std(v_sub) * 0.18))
        min_s = max(5, min(20, eff_chunk // 4))
        db = DBSCAN(eps=eps, min_samples=min_s, metric="euclidean")
        labels = db.fit_predict(v_sub.reshape(-1, 1))
        for lab in [l for l in np.unique(labels) if l != -1]:
            c_local = np.where(labels == lab)[0]
            if len(c_local) >= eff_chunk:
                srt = c_local[np.argsort(v_sub[c_local])]
                for ci in range(len(srt) // eff_chunk):
                    chunk_list.append(candidate_idx[srt[ci * eff_chunk:(ci + 1) * eff_chunk]])
                    if len(chunk_list) >= num_chunks:
                        break
            if len(chunk_list) >= num_chunks:
                break
        assigned = set(np.concatenate(chunk_list)) if chunk_list else set()
        candidate_idx = np.array([i for i in candidate_idx if i not in assigned])

    # Fallback: sequential fill
    if len(chunk_list) < num_chunks:
        assigned = set(np.concatenate(chunk_list)) if chunk_list else set()
        avail = [i for i in range(dim) if i not in assigned and i not in super_coords]
        for _ in range(num_chunks - len(chunk_list)):
            if len(avail) >= eff_chunk:
                chunk_list.append(np.array(avail[:eff_chunk]))
                avail = avail[eff_chunk:]
            else:
                break

    if not chunk_list:
        raise RuntimeError(f"Clustering failed: dim={dim}, eff_chunk={eff_chunk}")

    # Build 3D tensor (num_chunks, eff_chunk, hidden_dim)
    # act_matrix rows = tokens, cols = coordinates
    # We take slices along the coordinate axis
    T = np.stack([act_matrix[:, c] for c in chunk_list], axis=0).transpose(0, 2, 1)
    # T shape: (K, eff_chunk, N_tokens) — but we want (K, chunk_size, hidden)
    # Better: stack coordinate slices of the FULL activation matrix
    # T[k] = act_matrix[:, chunk_list[k]]  => shape (N_tokens, eff_chunk)
    # Reshape to (K, N_tokens, eff_chunk) => Tucker on that
    T_torch = torch.stack([
        torch.tensor(act_matrix[:, c], dtype=torch.float32) for c in chunk_list
    ], dim=0)  # (K, N_tokens, eff_chunk)

    return {
        "tensor": T_torch,
        "chunk_list": chunk_list,
        "super_coords": super_coords,
        "eff_chunk": eff_chunk,
        "num_chunks": len(chunk_list),
        "dim": dim,
    }



In [ ]:
# =====================================================================
# STEP 5b: Weight Matrix Clustering for Linear Projection Submodules
# =====================================================================
def nested_clustering_weight(
    act_matrix: np.ndarray,   # activation (N_tokens, out_dim) for guidance
    weight_tensor: torch.Tensor,  # weight shape (out_dim, in_dim)
    chunk_size: int,
    num_chunks: int,
    n_iter: int = 3,
    z_cutoff: float = 3.0,
) -> Dict[str, Any]:
    """
    For Linear projections: clusters the output-coordinate (row) axis of the weight
    matrix guided by activation statistics. Returns a 3D weight tensor.
    T shape: (K, chunk_size, in_dim)
    """
    out_dim, in_dim = weight_tensor.shape

    # Use activation output statistics to guide clustering
    if act_matrix is not None and act_matrix.shape[1] == out_dim:
        v = np.mean(act_matrix, axis=0)
        variances = np.var(act_matrix, axis=0)
    else:
        w_np = weight_tensor.detach().cpu().float().numpy()
        v = np.mean(w_np, axis=1)
        variances = np.var(w_np, axis=1)

    z = np.abs((v - np.mean(v)) / (np.std(v) + 1e-8))
    var99 = float(np.quantile(variances, 0.99)) if out_dim > 10 else 1e9
    super_mask = (z > z_cutoff) | (variances >= var99)
    super_coords = np.where(super_mask)[0]

    candidate_idx = np.where(~super_mask)[0]
    eff_chunk = chunk_size
    if len(candidate_idx) < num_chunks * eff_chunk:
        eff_chunk = max(5, len(candidate_idx) // num_chunks)

    chunk_list = []

    for _ in range(n_iter):
        if len(candidate_idx) < eff_chunk or len(chunk_list) >= num_chunks:
            break
        v_sub = v[candidate_idx]
        eps = max(0.02, float(np.std(v_sub) * 0.18))
        min_s = max(5, min(20, eff_chunk // 4))
        db = DBSCAN(eps=eps, min_samples=min_s, metric="euclidean")
        labels = db.fit_predict(v_sub.reshape(-1, 1))
        for lab in [l for l in np.unique(labels) if l != -1]:
            c_local = np.where(labels == lab)[0]
            if len(c_local) >= eff_chunk:
                srt = c_local[np.argsort(v_sub[c_local])]
                for ci in range(len(srt) // eff_chunk):
                    chunk_list.append(candidate_idx[srt[ci * eff_chunk:(ci + 1) * eff_chunk]])
                    if len(chunk_list) >= num_chunks:
                        break
            if len(chunk_list) >= num_chunks:
                break
        assigned = set(np.concatenate(chunk_list)) if chunk_list else set()
        candidate_idx = np.array([i for i in candidate_idx if i not in assigned])

    if len(chunk_list) < num_chunks:
        assigned = set(np.concatenate(chunk_list)) if chunk_list else set()
        avail = [i for i in range(out_dim) if i not in assigned and i not in super_coords]
        for _ in range(num_chunks - len(chunk_list)):
            if len(avail) >= eff_chunk:
                chunk_list.append(np.array(avail[:eff_chunk]))
                avail = avail[eff_chunk:]
            else:
                break

    if not chunk_list:
        raise RuntimeError(f"Weight clustering failed: out_dim={out_dim}, eff_chunk={eff_chunk}")

    # Build 3D weight tensor (K, eff_chunk, in_dim)
    T = torch.stack([weight_tensor[c, :].float().cpu() for c in chunk_list], dim=0)

    return {
        "tensor": T,
        "chunk_list": chunk_list,
        "super_coords": super_coords,
        "eff_chunk": eff_chunk,
        "num_chunks": len(chunk_list),
        "out_dim": out_dim,
        "in_dim": in_dim,
    }



In [ ]:
# =====================================================================
# STEP 6: 1% Incremental Reconstruction Rate Loss Sweep
# =====================================================================
def bespoke_ranks_by_energy(T: torch.Tensor, tau: float) -> List[int]:
    """Finds Tucker ranks [R1, R2, R3] that retain fraction tau of spectral energy per mode."""
    ranks = []
    for mode in range(T.dim()):
        unfolded = tl.unfold(T, mode)
        s = torch.linalg.svdvals(unfolded)
        cum_e = torch.cumsum(s**2, 0) / s.pow(2).sum()
        idx = (cum_e >= tau).nonzero()
        r = int(idx[0].item()) + 1 if len(idx) > 0 else T.shape[mode]
        r = max(1, min(T.shape[mode], r))
        ranks.append(r)
    return ranks

def sweep_1pct(T: torch.Tensor, start=0.99, end=0.70, step=0.01) -> List[Dict]:
    """Steps tau in 1% decrements, computing Tucker ranks, recon error, and param cut."""
    candidates = []
    tau = start
    while tau >= end - 1e-6:
        ranks = bespoke_ranks_by_energy(T, tau)
        core, factors = tucker(T, rank=ranks, init="svd")
        T_hat = tucker_to_tensor((core, factors))
        rel_err = (torch.norm(T - T_hat) / torch.norm(T)).item()
        orig_p = T.numel()
        comp_p = core.numel() + sum(f.numel() for f in factors)
        cut_pct = (orig_p - comp_p) / orig_p * 100.0
        candidates.append({
            "tau": round(tau, 4),
            "loss_pct": round((1.0 - tau) * 100.0, 2),
            "ranks": ranks,
            "recon_err_pct": round(rel_err * 100.0, 2),
            "params_cut_pct": round(cut_pct, 2),
            "T_hat": T_hat,
        })
        tau -= step
    return candidates



In [ ]:
# =====================================================================
# STEP 7: Tucker Adaptation Loop Across the 8 Target Submodules
# =====================================================================
#
# Tucker mode:
#   "weight"  — Linear projections: Tucker on weight matrix, inject back into model weights
#   "activation" — Container/norm modules: Tucker on output activation tensor (analysis only)
#
SUBMODULE_CONFIGS = {
    # --- Container modules: Tucker on output activation ---
    "self_attn": {
        "mode": "activation",
        "act_key": "self_attn",
        "desc": "Output Shape: (5000, 1152)",
        "chunk_size": 250, "num_chunks": 6,
    },
    # --- Linear projections: Tucker on weight matrix ---
    "self_attn.q_proj": {
        "mode": "weight",
        "module": l0.self_attn.q_proj,
        "act_key": "self_attn.q_proj",
        "desc": "Weight [1024, 1152] | Output Shape: (5000, 1024)",
        "chunk_size": 240, "num_chunks": 4,
    },
    "self_attn.k_proj": {
        "mode": "weight",
        "module": l0.self_attn.k_proj,
        "act_key": "self_attn.k_proj",
        "desc": "Weight [256, 1152] | Output Shape: (5000, 256)",
        "chunk_size": 60, "num_chunks": 4,
    },
    "self_attn.v_proj": {
        "mode": "weight",
        "module": l0.self_attn.v_proj,
        "act_key": "self_attn.v_proj",
        "desc": "Weight [256, 1152] | Output Shape: (5000, 256)",
        "chunk_size": 60, "num_chunks": 4,
    },
    "self_attn.o_proj": {
        "mode": "weight",
        "module": l0.self_attn.o_proj,
        "act_key": "self_attn.o_proj",
        "desc": "Weight [1152, 1024] | Output Shape: (5000, 1152)",
        "chunk_size": 250, "num_chunks": 4,
    },
    # --- 1D RMSNorm: Tucker on output activation ---
    "self_attn.q_norm": {
        "mode": "activation",
        "act_key": "self_attn.q_norm",
        "desc": "1D RMSNorm scale [256] | Output Shape: (5000, 256)",
        "chunk_size": 60, "num_chunks": 4,
    },
    "self_attn.k_norm": {
        "mode": "activation",
        "act_key": "self_attn.k_norm",
        "desc": "1D RMSNorm scale [256] | Output Shape: (5000, 256)",
        "chunk_size": 60, "num_chunks": 4,
    },
    # --- Container module: Tucker on output activation ---
    "mlp": {
        "mode": "activation",
        "act_key": "mlp",
        "desc": "Output Shape: (5000, 1152)",
        "chunk_size": 250, "num_chunks": 6,
    },
}

all_results = {}

for sub_name, cfg in SUBMODULE_CONFIGS.items():
    print("\n" + "=" * 80)
    print(f"TUCKER ADAPTATION: {sub_name}  [{cfg['mode'].upper()} MODE]")
    print(f"  {cfg['desc']}")
    print("=" * 80)

    act = acts[cfg["act_key"]]   # np.ndarray (N_tokens, dim)

    # ----------------------------------------------------------------
    # Build 3D tensor T
    # ----------------------------------------------------------------
    if cfg["mode"] == "weight":
        mod = cfg["module"]
        W_orig = mod.weight.data.clone()
        cdata = nested_clustering_weight(
            act_matrix=act,
            weight_tensor=W_orig,
            chunk_size=cfg["chunk_size"],
            num_chunks=cfg["num_chunks"],
        )
    else:  # activation mode
        W_orig = None
        cdata = nested_clustering_with_thresholding(
            act_matrix=act,
            chunk_size=cfg["chunk_size"],
            num_chunks=cfg["num_chunks"],
        )

    T = cdata["tensor"]
    n_super = len(cdata["super_coords"])
    n_dim = cdata.get("out_dim", cdata.get("dim", act.shape[1]))
    print(f"3D Tensor: {list(T.shape)}  |  Superweight/superact coords quarantined: {n_super}/{n_dim} ({n_super/n_dim*100:.1f}%)")

    # ----------------------------------------------------------------
    # 1% incremental reconstruction sweep
    # ----------------------------------------------------------------
    candidates = sweep_1pct(T, start=0.99, end=0.70, step=0.01)
    print(f"Generated {len(candidates)} candidate rank compositions (1% steps).")

    sweep_log = []
    best_acc = -1.0
    best_entry = None
    best_cand = None

    print(f"\n{'Loss %':>7} | {'Tau':>6} | {'Ranks':<15} | {'Recon Err%':>10} | {'Cut%':>7}", end="")

    if cfg["mode"] == "weight":
        print(f" | {'MNLI Acc%':>9}")
    else:
        print()  # No downstream injection for activation mode; still measure recon quality

    print("-" * (75 if cfg["mode"] == "weight" else 55))

    for cand in candidates:
        if cfg["mode"] == "weight":
            # Inject Tucker reconstruction into weight matrix
            mod.weight.data = W_orig.clone()
            T_hat = cand["T_hat"]
            for k_idx, c in enumerate(cdata["chunk_list"]):
                mod.weight.data[c, :] = T_hat[k_idx].to(mod.weight.device, dtype=mod.weight.dtype)

            acc = evaluate_mnli(model)
            mod.weight.data = W_orig.clone()  # Restore

            entry = {
                "loss_pct": cand["loss_pct"], "tau": cand["tau"],
                "ranks": cand["ranks"], "recon_err_pct": cand["recon_err_pct"],
                "params_cut_pct": cand["params_cut_pct"],
                "accuracy_pct": round(acc * 100.0, 2),
            }
            sweep_log.append(entry)
            if acc > best_acc:
                best_acc = acc
                best_entry = entry
                best_cand = cand

            print(f"{cand['loss_pct']:>7.1f} | {cand['tau']:>6.2f} | {str(cand['ranks']):<15} | "
                  f"{cand['recon_err_pct']:>10.2f} | {cand['params_cut_pct']:>7.2f} | {acc*100:>9.2f}")
        else:
            # Activation mode: record recon quality only (no downstream injection)
            entry = {
                "loss_pct": cand["loss_pct"], "tau": cand["tau"],
                "ranks": cand["ranks"], "recon_err_pct": cand["recon_err_pct"],
                "params_cut_pct": cand["params_cut_pct"],
                "accuracy_pct": None,
            }
            sweep_log.append(entry)
            if best_cand is None or cand["recon_err_pct"] < best_entry["recon_err_pct"]:
                best_entry = entry
                best_cand = cand

            print(f"{cand['loss_pct']:>7.1f} | {cand['tau']:>6.2f} | {str(cand['ranks']):<15} | "
                  f"{cand['recon_err_pct']:>10.2f} | {cand['params_cut_pct']:>7.2f}")

    # ----------------------------------------------------------------
    # Qualitative generation at best composition (weight mode only)
    # ----------------------------------------------------------------
    adapted_cake = ""
    if cfg["mode"] == "weight" and best_cand is not None:
        print(f"\n--- Chocolate Cake Recipe @ Best Ranks {best_entry['ranks']} ---")
        mod.weight.data = W_orig.clone()
        T_hat = best_cand["T_hat"]
        for k_idx, c in enumerate(cdata["chunk_list"]):
            mod.weight.data[c, :] = T_hat[k_idx].to(mod.weight.device, dtype=mod.weight.dtype)
        adapted_cake = generate_cake_recipe(model)
        print(adapted_cake[:350] + "...\n")
        mod.weight.data = W_orig.clone()  # Final restore

    print(f"\n>> BEST for {sub_name}: Ranks={best_entry['ranks']}", end="")
    if cfg["mode"] == "weight":
        print(f"  Acc={best_entry['accuracy_pct']:.2f}%  (Baseline: {baseline_acc*100:.2f}%)", end="")
    print(f"  ReconErr={best_entry['recon_err_pct']:.2f}%  ParamCut={best_entry['params_cut_pct']:.2f}%")

    all_results[sub_name] = {
        "mode": cfg["mode"],
        "desc": cfg["desc"],
        "tensor_shape": list(T.shape),
        "quarantined_coords": int(n_super),
        "quarantined_pct": round(n_super / n_dim * 100, 2),
        "best_composition": best_entry,
        "cake_recipe": adapted_cake,
        "full_sweep": sweep_log,
    }



In [ ]:
# =====================================================================
# STEP 8: Visualizations — Pareto Frontier & Reconstruction Error Curves
# =====================================================================
weight_subs = {k: v for k, v in all_results.items() if v["mode"] == "weight"}
act_subs    = {k: v for k, v in all_results.items() if v["mode"] == "activation"}

fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=130)

# --- Linear Projections: Accuracy vs Reconstruction Loss ---
ax = axes[0, 0]
for name, data in weight_subs.items():
    sweep = data["full_sweep"]
    ax.plot([s["loss_pct"] for s in sweep], [s["accuracy_pct"] for s in sweep],
            marker="o", markersize=3, label=name)
ax.axhline(baseline_acc * 100, color="k", linestyle="--", lw=1.5, label="Pristine Baseline")
ax.set_xlabel("Recon Energy Loss (%)")
ax.set_ylabel("MNLI Accuracy (%)")
ax.set_title("Linear Projections: Accuracy vs Recon Loss (1% steps)", fontweight="bold")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(fontsize=8)

# --- Linear Projections: Accuracy vs Parameter Cut ---
ax = axes[0, 1]
for name, data in weight_subs.items():
    sweep = data["full_sweep"]
    ax.plot([s["params_cut_pct"] for s in sweep], [s["accuracy_pct"] for s in sweep],
            marker="s", markersize=3, label=name)
ax.axhline(baseline_acc * 100, color="k", linestyle="--", lw=1.5, label="Pristine Baseline")
ax.set_xlabel("Parameter Reduction (%)")
ax.set_ylabel("MNLI Accuracy (%)")
ax.set_title("Linear Projections: Pareto Frontier (Accuracy vs Compression)", fontweight="bold")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(fontsize=8)

# --- Activation Modules: Reconstruction Error vs Loss % ---
ax = axes[1, 0]
for name, data in act_subs.items():
    sweep = data["full_sweep"]
    ax.plot([s["loss_pct"] for s in sweep], [s["recon_err_pct"] for s in sweep],
            marker="^", markersize=3, label=name)
ax.set_xlabel("Target Recon Energy Loss (%)")
ax.set_ylabel("Actual Recon Error (%)")
ax.set_title("Activation Modules: Recon Error Curve (self_attn, q_norm, k_norm, mlp)", fontweight="bold")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(fontsize=8)

# --- Activation Modules: Reconstruction Error vs Parameter Cut ---
ax = axes[1, 1]
for name, data in act_subs.items():
    sweep = data["full_sweep"]
    ax.plot([s["params_cut_pct"] for s in sweep], [s["recon_err_pct"] for s in sweep],
            marker="D", markersize=3, label=name)
ax.set_xlabel("Parameter Reduction (%)")
ax.set_ylabel("Actual Recon Error (%)")
ax.set_title("Activation Modules: Recon Error vs Compression", fontweight="bold")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("experiments/01_layer_based_bench/artifacts/10_layer_0_adaptation_pareto.png", dpi=150)
plt.show()
print("Saved: 10_layer_0_adaptation_pareto.png")



In [ ]:
# =====================================================================
# STEP 9: Final Summary Table & JSON Artifact Export
# =====================================================================
print("\n" + "=" * 100)
print("LAYER 0 TUCKER ADAPTATION — FINAL SUMMARY")
print(f"Pristine Baseline MNLI Accuracy: {baseline_acc * 100:.2f}%")
print("=" * 100)
print(f"{'Submodule':<24} | {'Mode':<12} | {'Tensor Shape':<20} | {'Best Ranks':<18} | {'Cut%':>7} | {'ReconErr%':>10} | {'MNLI Acc%':>10}")
print("-" * 108)
for name, data in all_results.items():
    bc = data["best_composition"]
    acc_str = f"{bc['accuracy_pct']:>10.2f}" if bc["accuracy_pct"] is not None else f"{'N/A':>10}"
    print(f"{name:<24} | {data['mode']:<12} | {str(data['tensor_shape']):<20} | "
          f"{str(bc['ranks']):<18} | {bc['params_cut_pct']:>7.2f} | {bc['recon_err_pct']:>10.2f} | {acc_str}")
print("=" * 108)

# JSON export
os.makedirs("experiments/01_layer_based_bench/artifacts", exist_ok=True)
payload = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model_id": MODEL_ID,
    "eval_samples": EVAL_SAMPLES,
    "baseline_accuracy_pct": round(baseline_acc * 100.0, 2),
    "cake_query_prompt": CAKE_PROMPT,
    "baseline_cake_recipe": baseline_cake,
    "target_submodules": list(SUBMODULE_CONFIGS.keys()),
    "submodule_results": {
        k: {
            "mode": v["mode"],
            "desc": v["desc"],
            "tensor_shape": v["tensor_shape"],
            "quarantined_coords": v["quarantined_coords"],
            "quarantined_pct": v["quarantined_pct"],
            "best_composition": v["best_composition"],
            "cake_recipe": v["cake_recipe"],
            "full_sweep": v["full_sweep"],
        }
        for k, v in all_results.items()
    }
}

out = "experiments/01_layer_based_bench/artifacts/10_layer_0_submodule_adaptation_results.json"
with open(out, "w") as f:
    json.dump(payload, f, indent=2)
print(f"\nResults saved to: {out}")

